# <span style='color:#00695c'>🧠 DataPrep — Saúde Mental no Trabalho Tech</span>
## Preparação de Dados · OSMI Mental Health in Tech Survey

---

| Projeto | Saúde Mental no Trabalho Tech |
|---------|------------------------------|
| Dataset | `data/processed/survey_limpo.csv` |
| Respondentes | 1.251 |
| Colunas | 24 |
| Casos de Uso | Classificação + Regressão |
| Equipe | Alan Lima · Antonio Feitosa · Gabriel Arthur · Heloiza Mendes · Michael Jhonathan |

> **Objetivo:** Preparar os dados do survey OSMI para dois modelos de Machine Learning:
> - **Classificação:** Prever risco de interferência da saúde mental no trabalho (`work_interfere`)
> - **Regressão:** Prever a faixa etária de busca por tratamento (`Age`) baseada no ambiente organizacional

---
## ⚙️ Seção 0 — Setup e Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Tenta importar sweetviz (opcional)
try:
    import sweetviz as sv
    HAS_SWEETVIZ = True
except ImportError:
    HAS_SWEETVIZ = False
    print("⚠️  sweetviz não instalado. Execute: pip install sweetviz")

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

CSV_PATH = 'data/processed/survey_limpo.csv'
df = pd.read_csv(CSV_PATH)

print(f'✅ Dataset carregado — Shape: {df.shape}')
print(f'   Colunas: {list(df.columns)}')

---
## 📋 Seção 1 — Questionário Problema-Solução

> Resposta às **17 perguntas de negócio** para os dois casos de uso,
> com foco em **burnout**, retenção de talentos e proatividade do RH.

### 🔵 Caso 1 — Classificação: Prever Risco de Interferência (`work_interfere`)

| # | Pergunta | Resposta |
|---|----------|----------|
| 1 | **O que queremos prever?** | Se um profissional de tech apresenta risco **alto** (Often/Sometimes) ou **baixo** (Rarely/Never) de ter sua saúde mental interferindo no trabalho. |
| 2 | **Por que isso é importante para o negócio?** | Permite ao RH agir preventivamente antes do burnout, reduzindo turnover, afastamentos e perda de produtividade. |
| 3 | **Quem é o usuário final desta previsão?** | Gestores de RH e líderes de time que precisam identificar grupos de risco sem exposição individual dos funcionários. |
| 4 | **Como a previsão será usada?** | Para segmentar equipes por nível de risco e acionar programas de apoio, revisão de cultura e ajustes no regime de trabalho. |
| 5 | **Qual é o custo de um erro (falso negativo)?** | **Alto.** Um funcionário classificado como baixo risco que na verdade está em burnout não recebe suporte — risco de afastamento e desligamento. |
| 6 | **Qual é a métrica de sucesso prioritária?** | **F1-Score (macro)** — equilibra precisão e recall, crucial dado o desbalanceamento entre as classes. |
| 7 | **Quais dados estão disponíveis?** | 1.251 respondentes com variáveis comportamentais e de ambiente organizacional do OSMI Survey 2014. |
| 8 | **Qual é a variável-alvo?** | `work_interfere` (binarizada): **1 = Alto Risco** (Often/Sometimes), **0 = Baixo Risco** (Rarely/Never). |
| 9 | **Quais são as features candidatas?** | `remote_work`, `anonymity`, `leave`, `no_employees`, `supervisor`, `family_history`, `treatment`, `benefits`, `coworkers`. |
| 10 | **Existe risco de vazamento de dados (data leakage)?** | Sim — `treatment` pode ser pós-diagnóstico e correlacionar diretamente com o target. Avaliar exclusão ou uso com cautela. |
| 11 | **Qual é o volume de dados?** | 989 registros válidos após remover os 262 nulos do target (`work_interfere`). Volume suficiente para treino/teste. |
| 12 | **Os dados estão balanceados?** | Parcialmente: ~62% Alto Risco vs ~38% Baixo Risco. Aplicar `class_weight='balanced'` no modelo. |
| 13 | **Quais são os valores ausentes relevantes?** | `work_interfere`: 262 nulos (21%) — **linhas removidas** para o dataset de classificação. `self_employed`: 18 nulos. |
| 14 | **Existe multicolinearidade entre features?** | Provável entre `anonymity` e `mental_health_consequence`. Verificar VIF após encoding. |
| 15 | **Qual é a frequência de atualização dos dados?** | Dataset histórico (2014). Para uso em produção, recomenda-se pesquisa interna anual. |
| 16 | **Existem restrições legais ou éticas?** | Dados sensíveis de saúde mental. **Não usar variáveis de diagnóstico pessoal** — focar em variáveis organizacionais. |
| 17 | **Qual é o horizonte temporal da predição?** | Diagnóstico pontual por funcionário/equipe. Reavaliação recomendada a cada ciclo de pesquisa (trimestral/anual). |

### 🟠 Caso 2 — Regressão: Prever Idade de Busca por Tratamento (`Age`)

| # | Pergunta | Resposta |
|---|----------|----------|
| 1 | **O que queremos prever?** | A **idade** em que um profissional de tech busca tratamento para saúde mental, dado o perfil organizacional da empresa onde trabalha. |
| 2 | **Por que isso é importante para o negócio?** | Permite ao RH identificar faixas etárias críticas para intervenção proativa, otimizando benefícios e programas de bem-estar por geração. |
| 3 | **Quem é o usuário final desta previsão?** | Equipes de People Analytics e benefícios, para segmentação de programas de saúde por faixa etária. |
| 4 | **Como a previsão será usada?** | Para calibrar quando e como oferecer recursos de saúde mental — ex.: programas específicos para profissionais entre 25-35 anos. |
| 5 | **Qual é o custo de um erro (alta RMSE)?** | **Médio.** Uma estimativa de idade muito desviada pode direcionar recursos para a faixa etária errada, mas não causa dano direto. |
| 6 | **Qual é a métrica de sucesso prioritária?** | **RMSE** e **R²** — avaliam desvio absoluto e capacidade explanatória do modelo sobre a variância da idade. |
| 7 | **Quais dados estão disponíveis?** | 1.233 respondentes com `treatment=Yes` após filtragem e imputação (coluna `Age` sem nulos no dataset original). |
| 8 | **Qual é a variável-alvo?** | `Age` — variável numérica contínua (intervalo realista: 18–75 anos). |
| 9 | **Quais são as features candidatas?** | `no_employees`, `remote_work`, `benefits`, `care_options`, `wellness_program`, `seek_help`, `anonymity`, `leave`, `family_history`. |
| 10 | **Existe risco de vazamento de dados?** | Baixo. `Age` é uma característica demográfica independente do ambiente organizacional. |
| 11 | **Qual é o volume de dados?** | 1.251 registros (full dataset). Após imputação de nulos nas features, todos os registros são usados. |
| 12 | **A distribuição do target é normal?** | Levemente assimétrica à direita (média ~32 anos, desvio ~7). Verificar transformação log se necessário. |
| 13 | **Quais são os valores ausentes relevantes?** | `self_employed`: 18 nulos — imputados com **Moda**. Demais features categóricas também imputadas com Moda. |
| 14 | **Existe multicolinearidade entre features?** | Provável entre `benefits`, `care_options` e `wellness_program` (políticas relacionadas). Avaliar durante modelagem. |
| 15 | **Qual é a frequência de atualização dos dados?** | Mesmo que o caso de classificação — dataset histórico de 2014. |
| 16 | **Existem restrições legais ou éticas?** | Prever idade baseado em ambiente organizacional é ético — não usa dados de saúde individual. |
| 17 | **Qual é o horizonte temporal da predição?** | Predição pontual para subsidiar decisão estratégica de RH. Não requer atualização em tempo real. |

---
## 🔍 Seção 2 — EDA Inicial

### 2.1 — Shape, Colunas e Primeiros Registros

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nColunas ({len(df.columns)}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
df.head(5)

In [ ]:
df.tail(5)

### 2.2 — Estatísticas Descritivas

In [ ]:
# Numéricas
print('=== ESTATÍSTICAS NUMÉRICAS ===')
display(df.describe())

In [ ]:
# Categóricas
print('=== ESTATÍSTICAS CATEGÓRICAS ===')
display(df.describe(include='object'))

### 2.3 — Função `column_summary`

In [ ]:
def column_summary(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Retorna um resumo detalhado de cada coluna: tipo, nulos, únicos e amostra de valores."""
    resumo = []
    for col in dataframe.columns:
        nulos = dataframe[col].isnull().sum()
        pct_nulos = nulos / len(dataframe) * 100
        unicos = dataframe[col].nunique()
        amostra = dataframe[col].dropna().unique()[:5].tolist()
        resumo.append({
            'Coluna':       col,
            'Tipo':         str(dataframe[col].dtype),
            'Nulos':        nulos,
            '% Nulos':      round(pct_nulos, 2),
            'Únicos':       unicos,
            'Amostra':      amostra,
        })
    return pd.DataFrame(resumo)


summary = column_summary(df)
display(summary)

### 2.4 — Relatório Sweetviz

In [ ]:
if HAS_SWEETVIZ:
    relatorio = sv.analyze(df)
    relatorio.show_html('data/processed/relatorio_sweetviz.html')
    print('✅ Relatório salvo em: data/processed/relatorio_sweetviz.html')
else:
    print('⚠️  sweetviz não disponível. Instale com: pip install sweetviz')

---
## 🛠️ Seção 3 — Tratamento de Qualidade

### 3.1 — Identificação e Confirmação de Tipos

In [ ]:
print('=== TIPOS DE DADOS ===')
print(df.dtypes)
print()

colunas_numericas    = df.select_dtypes(include=['int64','float64']).columns.tolist()
colunas_categoricas  = df.select_dtypes(include=['object']).columns.tolist()

print(f'Numéricas  ({len(colunas_numericas)}): {colunas_numericas}')
print(f'Categóricas ({len(colunas_categoricas)}): {colunas_categoricas}')

> **Conclusão de tipos:**
> - `Age` é a única variável **numérica** (int64) — será o target da Regressão.
> - As outras 23 colunas são **categóricas** (object). Três delas possuem escala ordinal implícita e serão mapeadas.

### 3.2 — Mapeamento de Variáveis Ordinais

In [ ]:
# ── Mapeamentos ordinais ────────────────────────────────────────────────────
MAPA_WORK_INTERFERE = {
    'Never':     0,
    'Rarely':    1,
    'Sometimes': 2,
    'Often':     3,
}

MAPA_LEAVE = {
    'Very easy':           0,
    'Somewhat easy':       1,
    "Don't know":          2,
    'Somewhat difficult':  3,
    'Very difficult':      4,
}

MAPA_NO_EMPLOYEES = {
    '1-5':            0,
    '6-25':           1,
    '26-100':         2,
    '100-500':        3,
    '500-1000':       4,
    'More than 1000': 5,
}

# Aplicar mapeamentos
df['work_interfere_ord'] = df['work_interfere'].map(MAPA_WORK_INTERFERE)
df['leave_ord']          = df['leave'].map(MAPA_LEAVE)
df['no_employees_ord']   = df['no_employees'].map(MAPA_NO_EMPLOYEES)

print('=== VERIFICAÇÃO DOS MAPEAMENTOS ORDINAIS ===')
print('\nwork_interfere → work_interfere_ord:')
print(df[['work_interfere','work_interfere_ord']].value_counts().reset_index().sort_values('work_interfere_ord'))

print('\nleave → leave_ord:')
print(df[['leave','leave_ord']].value_counts().reset_index().sort_values('leave_ord'))

print('\nno_employees → no_employees_ord:')
print(df[['no_employees','no_employees_ord']].value_counts().reset_index().sort_values('no_employees_ord'))

### 3.3 — Tratamento de Inconsistências: Coluna `Age`

In [ ]:
print('=== ANÁLISE DA COLUNA AGE ANTES DA LIMPEZA ===')
print(df['Age'].describe())
print(f'\nValores fora do intervalo [18, 75]:')
fora = df[(df['Age'] < 18) | (df['Age'] > 75)]
print(f'  Total: {len(fora)} registros')
if len(fora) > 0:
    print(fora['Age'].value_counts().head(10))

# Filtrar para intervalo realista de profissionais
df = df[(df['Age'] >= 18) & (df['Age'] <= 75)].copy()

print(f'\n✅ Após limpeza de Age [18–75]:')
print(df['Age'].describe())
print(f'  Registros restantes: {len(df)}')

### 3.4 — Geração dos Dois Datasets

#### 3.4.1 — `df_classificacao`: Remover nulos do target `work_interfere`

In [ ]:
# Dataset de Classificação: remove linhas onde work_interfere é nulo
df_classificacao = df.dropna(subset=['work_interfere']).copy()

print('=== DATASET DE CLASSIFICAÇÃO ===')
print(f'Shape: {df_classificacao.shape}')
print(f'Registros removidos (nulos em work_interfere): {len(df) - len(df_classificacao)}')
print(f'\nDistribuição do target (work_interfere):')
print(df_classificacao['work_interfere'].value_counts())
print(f'\nNulos restantes:')
nulos_clf = df_classificacao.isnull().sum()
print(nulos_clf[nulos_clf > 0])

#### 3.4.2 — `df_regressao`: Imputar nulos das features categóricas com Moda

In [ ]:
# Dataset de Regressão: usa todos os registros, imputa nulos com Moda
df_regressao = df.copy()

colunas_com_nulos = df_regressao.columns[df_regressao.isnull().any()].tolist()
print(f'Colunas com nulos antes da imputação: {colunas_com_nulos}')

for col in colunas_com_nulos:
    if df_regressao[col].dtype == 'object':
        moda = df_regressao[col].mode()[0]
        n_imputados = df_regressao[col].isnull().sum()
        df_regressao[col] = df_regressao[col].fillna(moda)
        print(f'  {col}: {n_imputados} nulos → imputados com moda="{moda}"')

print(f'\n=== DATASET DE REGRESSÃO ===')
print(f'Shape: {df_regressao.shape}')
print(f'Nulos restantes após imputação: {df_regressao.isnull().sum().sum()}')
print(f'\nDistribuição do target (Age):')
print(df_regressao['Age'].describe())

### 3.5 — Métricas de Completude Final

In [ ]:
def avaliacao_completude(dataframe: pd.DataFrame, nome: str) -> None:
    """Exibe um relatório de completude e nulos para o dataset fornecido."""
    total_celulas = dataframe.shape[0] * dataframe.shape[1]
    total_nulos   = dataframe.isnull().sum().sum()
    completude    = (1 - total_nulos / total_celulas) * 100

    nulos_por_col = dataframe.isnull().sum()
    colunas_com_nulos = nulos_por_col[nulos_por_col > 0]

    print(f'{'='*55}')
    print(f'  RELATÓRIO DE COMPLETUDE — {nome}')
    print(f'{'='*55}')
    print(f'  Linhas:          {dataframe.shape[0]:>6,}')
    print(f'  Colunas:         {dataframe.shape[1]:>6,}')
    print(f'  Total de células:{total_celulas:>6,}')
    print(f'  Total de nulos:  {total_nulos:>6,}')
    print(f'  Completude:      {completude:>6.2f}%')
    print(f'{'─'*55}')
    if len(colunas_com_nulos) == 0:
        print('  ✅ Nenhuma coluna com nulos.')
    else:
        print('  Colunas com nulos:')
        for col, qtd in colunas_com_nulos.items():
            pct = qtd / dataframe.shape[0] * 100
            print(f'    {col:35s}: {qtd:4d} ({pct:.1f}%)')
    print(f'{'='*55}\n')


avaliacao_completude(df_classificacao, 'df_classificacao')
avaliacao_completude(df_regressao,     'df_regressao')

---
## 📊 Seção 4 — Visualizações de Validação

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Gráfico 1: Distribuição do target de classificação
ordem_wi = ['Never', 'Rarely', 'Sometimes', 'Often']
wi_counts = df_classificacao['work_interfere'].value_counts().reindex(ordem_wi)
cores = ['#4db6ac', '#80cbc4', '#ffb74d', '#e57373']
axes[0].bar(wi_counts.index, wi_counts.values, color=cores, edgecolor='none')
axes[0].set_title('Distribuição: work_interfere\n(df_classificacao)')
axes[0].set_xlabel('Nível')
axes[0].set_ylabel('Contagem')
for i, v in enumerate(wi_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9)
axes[0].spines[['top','right']].set_visible(False)

# Gráfico 2: Distribuição do target de regressão
axes[1].hist(df_regressao['Age'], bins=20, color='#00897b', edgecolor='none', alpha=0.85)
axes[1].set_title('Distribuição: Age\n(df_regressao)')
axes[1].set_xlabel('Idade')
axes[1].set_ylabel('Frequência')
axes[1].spines[['top','right']].set_visible(False)

# Gráfico 3: Nulos por dataset
datasets = ['Original', 'df_classificacao', 'df_regressao']
nulos    = [
    df.isnull().sum().sum(),
    df_classificacao.isnull().sum().sum(),
    df_regressao.isnull().sum().sum()
]
cores_b = ['#90a4ae', '#4db6ac', '#00695c']
axes[2].bar(datasets, nulos, color=cores_b, edgecolor='none')
axes[2].set_title('Total de Nulos por Dataset')
axes[2].set_ylabel('Qtd. de Valores Nulos')
for i, v in enumerate(nulos):
    axes[2].text(i, v + 1, str(v), ha='center', fontsize=9)
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('data/processed/dataprep_validacao.png', bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo em: data/processed/dataprep_validacao.png')

---
## 💾 Seção 5 — Exportação dos Datasets

In [ ]:
PATH_CLF = 'data/processed/df_classificacao.csv'
PATH_REG = 'data/processed/df_regressao.csv'

df_classificacao.to_csv(PATH_CLF, index=False)
df_regressao.to_csv(PATH_REG, index=False)

print(f'✅ df_classificacao.csv salvo  → {PATH_CLF}')
print(f'   Shape: {df_classificacao.shape}')
print()
print(f'✅ df_regressao.csv salvo      → {PATH_REG}')
print(f'   Shape: {df_regressao.shape}')

---
## ✅ Seção 6 — Síntese do DataPrep

| Etapa | Ação Realizada | Resultado |
|-------|----------------|----------|
| **Tipos** | `Age` identificado como numérico; demais 23 colunas como categóricas | Correto |
| **Ordinais** | Mapeamento manual de `work_interfere` (0–3), `leave` (0–4), `no_employees` (0–5) | 3 colunas ordinais criadas |
| **Age (inconsistências)** | Filtragem para intervalo [18, 75] | Dataset dentro do realístico |
| **Nulos — Classificação** | Remoção de 262 linhas com `work_interfere` nulo | `df_classificacao`: 989 linhas |
| **Nulos — Regressão** | Imputação com Moda em colunas categóricas (`self_employed`) | `df_regressao`: 1.251 linhas, 0 nulos |
| **Exportação** | Salvos em `data/processed/` | 2 arquivos prontos para ML |

---

**Próximos Passos:**
1. Encoding das features categóricas (OneHot ou Label, conforme modelo)
2. Feature selection (correlação, VIF)
3. Split treino/teste estratificado
4. Treinamento dos modelos de Classificação (Random Forest, Logistic Regression) e Regressão (Ridge, Random Forest Regressor)